# Conditional Chest X-ray VAE

A small end-to-end example of the PyHealth 2.0 image VAE. It runs on generated grayscale images so you can try the model without downloading clinical data. For real-data training, see `examples/cxr/chextXray_image_generation_VAE.py`.


## Create a tiny image dataset

`ImageProcessor` loads each image and scales its pixels to `[0, 1]`. The disease label is used as the VAE's generation condition.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw
import torch

from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.models import VAE
from pyhealth.trainer import Trainer

image_dir = TemporaryDirectory()
samples = []
labels = ["normal", "covid", "opacity"]
for index in range(12):
    label = labels[index % len(labels)]
    pixels = np.full((32, 32), 35 + 45 * (index % 3), dtype=np.uint8)
    image = Image.fromarray(pixels)
    draw = ImageDraw.Draw(image)
    draw.ellipse((6 + index % 5, 8, 25, 27), outline=220, width=2)
    path = Path(image_dir.name) / f"cxr-{index}.png"
    image.save(path)
    samples.append({"image": str(path), "disease": label})

dataset = create_sample_dataset(
    samples=samples,
    input_schema={"image": ("image", {"image_size": 32, "mode": "L"})},
    output_schema={"disease": "multiclass"},
    dataset_name="vae-cxr-demo",
)
loader = get_dataloader(dataset, batch_size=4, shuffle=True, seed=7)
batch = next(iter(loader))
print("Batch images:", batch["image"].shape)
print("Encoded labels:", batch["disease"].tolist())

## Train the image VAE

The image feature and label processor are inferred from the dataset. The trainer optimizes reconstruction plus KL-divergence loss.


In [ ]:
model = VAE(dataset, latent_dim=8, hidden_dim=4)
trainer = Trainer(
    model=model,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
trainer.train(train_dataloader=loader, epochs=2, optimizer_params={"lr": 1e-3})

## Inspect reconstructions


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for index in range(4):
    axes[0, index].imshow(batch["image"][index, 0].cpu(), cmap="gray")
    axes[0, index].set_title("Input")
    axes[0, index].axis("off")
    axes[1, index].imshow(
        result["reconstruction"][index, 0].cpu(), cmap="gray"
    )
    axes[1, index].set_title("Reconstruction")
    axes[1, index].axis("off")
plt.tight_layout()


## Generate images by class

`generate` accepts the encoded class indices learned by the fitted multiclass label processor.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for index in range(4):
    axes[0, index].imshow(batch["image"][index, 0].cpu(), cmap="gray")
    axes[0, index].set_title("Input")
    axes[0, index].axis("off")
    axes[1, index].imshow(
        result["reconstruction"][index, 0].cpu(), cmap="gray"
    )
    axes[1, index].set_title("Reconstruction")
    axes[1, index].axis("off")
plt.tight_layout()
